<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 15 · Machine and Deep Learning
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the current chapter-15 draft with executable examples
for classical machine learning, unsupervised learning, and PyTorch sequence
models.


### How to Use This Notebook
- Run the cells from top to bottom the first time so later sections can
reuse earlier imports and datasets.
- Keep the training and plotting cells together when you compare models.
- Use the chapter text for the surrounding interpretation and caveats.


Machine learning provides a flexible toolkit for finding patterns in data and
building predictive models. This notebook follows the draft chapter from
`scikit-learn` baselines to PyTorch neural networks and simple sequence
models.


## Supervised Learning with scikit-learn


Supervised learning starts from labeled examples and learns a mapping from
features to targets.


### Binary Classification on a Two-Moons Dataset


The `make_moons()` helper generates a curved two-class problem that is
ideal for visualising decision boundaries.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})
# Generate 500 two-dimensional points with moderate label noise.
X, y = make_moons(n_samples=500, noise=0.25, random_state=2027)

In [ ]:
X.shape, y.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
# Split the data into reproducible training and test sets with stratified
# labels.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=2027,
    stratify=y,
)
clf = LogisticRegression(solver="lbfgs")

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
# Plot the logistic-regression decision surface and test points.
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]
zz = clf.predict_proba(grid)[:, 1].reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7.0, 4.5))
contour = ax.contourf(
    xx,
    yy,
    zz,
    levels=np.linspace(0.0, 1.0, 21),
    cmap="coolwarm",
    alpha=0.7,
)
fig.colorbar(contour, ax=ax).set_label("Class 1 probability")
ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_test,
    cmap="coolwarm",
    edgecolor="k",
    linewidth=0.4,
    alpha=0.9,
    label="Test points",
)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title(
    f"Logistic Regression on Two-Moons Dataset "
    f"(acc={accuracy_score(y_test, y_pred):.3f})"
)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Nonlinear SVM Classifiers on the Two-Moons Dataset


An RBF-kernel SVM can represent curved decision boundaries in the same feature
space.


In [ ]:
from sklearn.svm import SVC
svm = SVC(kernel="rbf", gamma="scale", C=1.0, probability=True)

In [ ]:
svm.fit(X_train, y_train)

In [ ]:
y_pred_svm = svm.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred_svm)

In [ ]:
# Plot the SVM decision surface on the same two-moons dataset.
zz_svm = svm.predict_proba(grid)[:, 1].reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7.0, 4.5))
contour = ax.contourf(
    xx,
    yy,
    zz_svm,
    levels=np.linspace(0.0, 1.0, 21),
    cmap="coolwarm",
    alpha=0.7,
)
fig.colorbar(contour, ax=ax).set_label("Class 1 probability (SVM)")
ax.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_test,
    cmap="coolwarm",
    edgecolor="k",
    linewidth=0.4,
    alpha=0.9,
    label="Test points",
)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title(
    f"RBF SVM on Two-Moons Dataset "
    f"(acc={accuracy_score(y_test, y_pred_svm):.3f})"
)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Regression with Linear and Tree-Based Models


A one-dimensional synthetic regression problem lets you compare rigid linear
fits with flexible tree-based ones.


In [ ]:
from sklearn.datasets import make_regression
# Create a noisy one-feature regression problem and keep the true slope.
X_reg, y_reg, coef = make_regression(
    n_samples=400,
    n_features=1,
    n_informative=1,
    noise=10.0,
    coef=True,
    random_state=2027,
)

In [ ]:
X_reg.shape, y_reg.shape, coef

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
# Split the regression data into training and test sets.
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.3,
    random_state=2027,
)
lin_reg = LinearRegression()
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    random_state=2027,
)

In [ ]:
lin_reg.fit(X_train_reg, y_train_reg)

In [ ]:
rf_reg.fit(X_train_reg, y_train_reg)

In [ ]:
y_lin = lin_reg.predict(X_test_reg)
y_rf = rf_reg.predict(X_test_reg)

In [ ]:
mean_squared_error(y_test_reg, y_lin), mean_squared_error(y_test_reg, y_rf)

In [ ]:
# Plot the test-set fits from the linear and random-forest regressors.
x_grid = np.linspace(X_reg.min() - 1.0, X_reg.max() + 1.0, 400).reshape(-1, 1)
y_lin_grid = lin_reg.predict(x_grid)
y_rf_grid = rf_reg.predict(x_grid)
fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.scatter(
    X_train_reg[:, 0],
    y_train_reg,
    color="tab:blue",
    alpha=0.6,
    label="Train",
    edgecolor="k",
    linewidth=0.3,
)
ax.scatter(
    X_test_reg[:, 0],
    y_test_reg,
    color="tab:orange",
    alpha=0.7,
    label="Test",
    edgecolor="k",
    linewidth=0.3,
)
ax.plot(
    x_grid[:, 0],
    y_lin_grid,
    color="tab:green",
    linewidth=2.0,
    label=(
        "Linear regression "
        f"(MSE={mean_squared_error(y_test_reg, y_lin):.1f})"
    ),
)
ax.plot(
    x_grid[:, 0],
    y_rf_grid,
    color="tab:red",
    linewidth=2.0,
    linestyle="--",
    label=f"Random forest (MSE={mean_squared_error(y_test_reg, y_rf):.1f})",
)
ax.set_xlabel("Feature")
ax.set_ylabel("Target")
ax.set_title("Linear vs Random-Forest Regression on Synthetic Data")
ax.legend(loc="best")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Unsupervised Learning: PCA and Clustering


Without labels, you can still look for lower-dimensional structure and cluster
assignments.


### PCA and K-means on Synthetic Blobs


Synthetic Gaussian blobs are convenient for illustrating PCA projections and
K-means labels.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
# Generate four three-dimensional Gaussian clusters.
X_blob, y_blob = make_blobs(
    n_samples=600,
    n_features=3,
    centers=4,
    cluster_std=1.3,
    random_state=2027,
)

In [ ]:
X_blob.shape, y_blob.shape

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=2027, n_init=10)
labels = kmeans.fit_predict(X_blob)
pca = PCA(n_components=2, random_state=2027)
X_pca = pca.fit_transform(X_blob)

In [ ]:
# Plot K-means labels in the original coordinates and in PCA space.
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0))
ax = axes[0]
ax.scatter(
    X_blob[:, 0],
    X_blob[:, 1],
    c=labels,
    cmap="tab10",
    s=15,
    alpha=0.8,
    edgecolor="k",
    linewidth=0.2,
)
ax.set_title("K-means Clusters (First Two Features)")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.grid(True, linestyle="--", alpha=0.3)
ax = axes[1]
ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    cmap="tab10",
    s=15,
    alpha=0.8,
    edgecolor="k",
    linewidth=0.2,
)
ax.set_title("K-means Clusters in First Two PC Scores")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Thinking in PyTorch


PyTorch combines tensors, modules, losses, and optimisers in a define-by-run
workflow.


### Tensors, Gradients, and Optimisers


A tiny gradient example makes the core PyTorch mechanics visible before moving
to larger models.


In [ ]:
import torch
# Create a tensor with gradient tracking enabled.
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [ ]:
y = (x**2).sum()

In [ ]:
y.backward()

In [ ]:
x.grad

In [ ]:
from torch import nn
# Fit a one-parameter linear model to the exact relation y = 2x.
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x_batch = torch.tensor([[0.0], [1.0], [2.0]])
y_batch = 2.0 * x_batch
loss_fn = nn.MSELoss()

In [ ]:
for epoch in range(100):
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = loss_fn(preds, y_batch)
    loss.backward()
    optimizer.step()

In [ ]:
model.weight.data, model.bias.data

### A Simple MLP Classifier on the Two-Moons Dataset


A compact MLP can learn a much more flexible boundary than the linear
baseline.


In [ ]:
import torch
from torch import nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
torch.manual_seed(2027)
# Rebuild the two-moons data for the PyTorch example.
X_nn, y_nn = make_moons(
    n_samples=600,
    noise=0.25,
    random_state=2027,
)
X_nn = X_nn.astype("float32")
y_nn = y_nn.astype("float32")
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn,
    y_nn,
    test_size=0.3,
    random_state=2027,
    stratify=y_nn,
)

In [ ]:
class MLP(nn.Module):
    # A one-hidden-layer MLP is enough to model the curved boundary.
    def __init__(self, in_features: int = 2, hidden_features: int = 32) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, 1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
device = torch.device("cpu")
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
X_train_t = torch.from_numpy(X_train_nn).to(device)
y_train_t = torch.from_numpy(y_train_nn).to(device).view(-1, 1)

In [ ]:
model.train()
for epoch in range(500):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test_nn).to(device)
    logits_test = model(X_test_t)
    probs_test = torch.sigmoid(logits_test).cpu().numpy().ravel()
    y_pred_nn = (probs_test >= 0.5).astype(int)
    acc_nn = (y_pred_nn == y_test_nn).mean()

In [ ]:
acc_nn

In [ ]:
# Plot the MLP decision surface on the two-moons test set.
x_min_nn, x_max_nn = X_nn[:, 0].min() - 0.5, X_nn[:, 0].max() + 0.5
y_min_nn, y_max_nn = X_nn[:, 1].min() - 0.5, X_nn[:, 1].max() + 0.5
xx_nn, yy_nn = np.meshgrid(
    np.linspace(x_min_nn, x_max_nn, 300),
    np.linspace(y_min_nn, y_max_nn, 300),
)
grid_nn = np.c_[xx_nn.ravel(), yy_nn.ravel()].astype("float32")
with torch.no_grad():
    logits_grid = model(torch.from_numpy(grid_nn).to(device))
    probs_grid = torch.sigmoid(logits_grid).cpu().numpy().ravel()
zz_nn = probs_grid.reshape(xx_nn.shape)
fig, ax = plt.subplots(figsize=(7.0, 4.5))
contour = ax.contourf(
    xx_nn,
    yy_nn,
    zz_nn,
    levels=np.linspace(0.0, 1.0, 21),
    cmap="coolwarm",
    alpha=0.7,
)
fig.colorbar(contour, ax=ax).set_label("Class 1 probability (MLP)")
ax.scatter(
    X_test_nn[:, 0],
    X_test_nn[:, 1],
    c=y_test_nn,
    cmap="coolwarm",
    edgecolor="k",
    linewidth=0.4,
    alpha=0.9,
    label="Test points",
)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title(f"PyTorch MLP on Two-Moons Dataset (acc={acc_nn:.3f})")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Sequence Models and Financial Time Series


Sequence models use lagged windows instead of isolated observations so that
recent history remains part of the input.


### From Prices to Lagged-Return Windows


You can build overlapping return windows and aligned targets from liquid
series in the end-of-day dataset.


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
# Load the end-of-day data relative to the notebook location.
data = pd.read_csv(
    "../data/eod_data.csv",
    index_col="Date",
    parse_dates=True,
).dropna()
# Configuration (symbol and sequence parameters).
SYMBOL_CLS = "BTC-USD"
WINDOW_CLS = 20
HORIZON_CLS = 5

prices_cls = data[SYMBOL_CLS].dropna()
log_returns_cls = np.log(prices_cls / prices_cls.shift(1)).dropna()
window_cls = WINDOW_CLS
horizon_cls = HORIZON_CLS

In [ ]:
# Create lagged return windows and forward-direction labels.
def build_lagged_direction_dataset(
    returns: pd.Series,
    window: int,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rets_local = returns.dropna().to_numpy(dtype=np.float32)
    dates_local = returns.dropna().index
    if rets_local.shape[0] <= window + horizon:
        raise ValueError(
            "Not enough observations for the chosen window and horizon."
        )
    X_list = []
    y_list = []
    date_list = []
    for t in range(window, rets_local.shape[0] - horizon + 1):
        X_list.append(rets_local[t - window : t].reshape(-1, 1))
        y_list.append(int(rets_local[t : t + horizon].sum() > 0.0))
        date_list.append(dates_local[t + horizon - 1])
    X_seq = np.stack(X_list, axis=0)
    y_dir = np.asarray(y_list, dtype=np.float32)
    return X_seq, y_dir, np.asarray(date_list)
X_seq, y_dir, dates_seq = build_lagged_direction_dataset(
    log_returns_cls,
    window=window_cls,
    horizon=horizon_cls,
)

In [ ]:
X_seq.shape, y_dir.shape

### LSTM Classification: Forward Up/Down


This notebook mirrors the sequence-classification figures from the chapter
assets (configure via `SYMBOL_CLS`, `WINDOW_CLS`, `HORIZON_CLS`).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Split the lagged windows without shuffling and scale them from the
# training sample only.
(
    X_train_seq,
    X_test_seq,
    y_train_seq,
    y_test_seq,
    dates_train_seq,
    dates_test_seq,
) = train_test_split(
    X_seq,
    y_dir,
    dates_seq,
    test_size=0.3,
    shuffle=False,
)
scaler_seq = StandardScaler()
X_train_flat = X_train_seq.reshape(X_train_seq.shape[0], -1)
X_test_flat = X_test_seq.reshape(X_test_seq.shape[0], -1)
X_train_scaled = scaler_seq.fit_transform(X_train_flat)
X_train_scaled = X_train_scaled.reshape(X_train_seq.shape)
X_test_scaled = scaler_seq.transform(X_test_flat).reshape(X_test_seq.shape)

In [ ]:
class LSTMClassifier(nn.Module):
    # Feed the final LSTM hidden state into a linear classifier.
    def __init__(self, input_size: int = 1, hidden_size: int = 16) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_out = out[:, -1, :]
        return self.fc(last_out)

In [ ]:
# Match the chapter figure script configuration for the sequence classifier.
device = torch.device("cpu")
torch.manual_seed(2027)
model_seq = LSTMClassifier(input_size=1, hidden_size=16).to(device)
optimizer_seq = torch.optim.Adam(model_seq.parameters(), lr=0.001)
pos_seq = float(y_train_seq.mean())
pos_weight_seq = (1.0 - pos_seq) / pos_seq
loss_fn_seq = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight_seq, dtype=torch.float32)
)
X_train_seq_t = torch.from_numpy(X_train_scaled).to(device)
y_train_seq_t = torch.from_numpy(y_train_seq).to(device).view(-1, 1)

In [ ]:
model_seq.train()
for epoch in range(2500):
    optimizer_seq.zero_grad()
    logits_seq = model_seq(X_train_seq_t)
    loss_seq = loss_fn_seq(logits_seq, y_train_seq_t)
    loss_seq.backward()
    optimizer_seq.step()

In [ ]:
model_seq.eval()
with torch.no_grad():
    X_test_seq_t = torch.from_numpy(X_test_scaled).to(device)
    logits_test_seq = model_seq(X_test_seq_t)
    probs_test_seq = torch.sigmoid(logits_test_seq).cpu().numpy().ravel()
    y_pred_seq = (probs_test_seq >= 0.5).astype(int)
    acc_seq = float((y_pred_seq == y_test_seq).mean())
cm_seq = confusion_matrix(y_test_seq, y_pred_seq, labels=[0, 1])
cm_seq_norm = cm_seq.astype(float) / cm_seq.sum(axis=1, keepdims=True)

In [ ]:
acc_seq

In [ ]:
# Plot daily returns and mark the train-test split used by the LSTM classifier.
fig, ax = plt.subplots(figsize=(9.5, 3.0))
ax.plot(
    log_returns_cls.index,
    log_returns_cls.values * 100.0,
    color="tab:blue",
    linewidth=0.8,
    label="Daily log return",
)
ax.axvline(
    dates_test_seq[0],
    color="k",
    linestyle="--",
    linewidth=1.0,
    alpha=0.7,
)
ax.set_ylabel("Daily log return (%)")
ax.set_title(f"{SYMBOL_CLS} daily returns with LSTM train/test split")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Plot the row-normalised confusion matrix in the same compact style as
# the figure script.
fig_cm, ax_cm = plt.subplots(figsize=(3.4, 2.6))
im = ax_cm.imshow(cm_seq_norm, cmap="Blues", vmin=0.0, vmax=1.0)
for i in range(2):
    for j in range(2):
        ax_cm.text(
            j,
            i,
            f"{cm_seq[i, j]}\n({cm_seq_norm[i, j]:.2f})",
            ha="center",
            va="center",
            color="black",
            fontsize=7,
        )
ax_cm.set_xticks([0, 1])
ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(["Down (0)", "Up (1)"], rotation=20, fontsize=7)
ax_cm.set_yticklabels(["Down (0)", "Up (1)"], fontsize=7)
ax_cm.set_xlabel("Predicted label", fontsize=7)
ax_cm.set_ylabel("True label", fontsize=7)
ax_cm.tick_params(axis="both", labelsize=7)
ax_cm.set_title(
    f"{SYMBOL_CLS} LSTM {horizon_cls}-step up/down (acc={acc_seq:.3f})",
    fontsize=8,
)
cbar = fig_cm.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
cbar.set_label("Row-normalised freq.", fontsize=7)
cbar.ax.tick_params(labelsize=7)
fig_cm.tight_layout(pad=0.8)
plt.show()

### LSTM Regression: Forward Return Prediction


This notebook mirrors the forward-return regression diagnostics and
equity-curve figures from the chapter assets (configure via `SYMBOL_REG`,
`WINDOW_REG`, `HORIZON_REG`).


In [ ]:
# Build lagged return windows and forward-return targets.
SYMBOL_REG = "SPY"
WINDOW_REG = 10
HORIZON_REG = 1

def build_lagged_regression_dataset(
    returns: pd.Series,
    window: int,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rets_local = returns.dropna().to_numpy(dtype=np.float32)
    dates_local = returns.dropna().index
    if rets_local.shape[0] <= window + horizon:
        raise ValueError(
            "Not enough observations for the chosen window and horizon."
        )
    X_list = []
    y_list = []
    date_list = []
    for t in range(window, rets_local.shape[0] - horizon + 1):
        X_list.append(rets_local[t - window : t].reshape(-1, 1))
        y_list.append(float(rets_local[t : t + horizon].sum()))
        date_list.append(dates_local[t + horizon - 1])
    X_reg_seq = np.stack(X_list, axis=0)
    y_reg_seq = np.asarray(y_list, dtype=np.float32)
    return X_reg_seq, y_reg_seq, np.asarray(date_list)
prices_reg = data[SYMBOL_REG].dropna()
log_returns_reg = np.log(prices_reg / prices_reg.shift(1)).dropna()
X_reg_seq, y_reg_seq, dates_reg = build_lagged_regression_dataset(
    log_returns_reg,
    window=WINDOW_REG,
    horizon=HORIZON_REG,
)

In [ ]:
class LSTMRegressor(nn.Module):
    # Use the final LSTM state to produce a single real-valued return forecast.
    def __init__(self, input_size: int = 1, hidden_size: int = 16) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_out = out[:, -1, :]
        return self.fc(last_out)

In [ ]:
# Match the chapter figure script configuration for the sequence regressor.
(
    X_train_reg_seq,
    X_test_reg_seq,
    y_train_reg_seq,
    y_test_reg_seq,
    dates_train_reg,
    dates_test_reg,
) = train_test_split(
    X_reg_seq,
    y_reg_seq,
    dates_reg,
    test_size=0.3,
    shuffle=False,
)
scaler_reg = StandardScaler()
X_train_reg_flat = X_train_reg_seq.reshape(X_train_reg_seq.shape[0], -1)
X_test_reg_flat = X_test_reg_seq.reshape(X_test_reg_seq.shape[0], -1)
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg_flat).reshape(
    X_train_reg_seq.shape
)
X_test_reg_scaled = scaler_reg.transform(X_test_reg_flat).reshape(
    X_test_reg_seq.shape
)
device = torch.device("cpu")
torch.manual_seed(2027)
model_reg = LSTMRegressor(input_size=1, hidden_size=16).to(device)
opt_reg = torch.optim.Adam(model_reg.parameters(), lr=0.01)
loss_reg = nn.MSELoss()
X_train_reg_t = torch.from_numpy(X_train_reg_scaled).to(device)
y_train_reg_t = torch.from_numpy(y_train_reg_seq).to(device).view(-1, 1)

In [ ]:
model_reg.train()
for epoch in range(2500):
    opt_reg.zero_grad()
    preds_reg_train = model_reg(X_train_reg_t)
    loss_reg_value = loss_reg(preds_reg_train, y_train_reg_t)
    loss_reg_value.backward()
    opt_reg.step()

In [ ]:
model_reg.eval()
with torch.no_grad():
    X_test_reg_t = torch.from_numpy(X_test_reg_scaled).to(device)
    preds_reg = model_reg(X_test_reg_t).cpu().numpy().ravel()
mse_reg = float(((preds_reg - y_test_reg_seq) ** 2).mean())
corr_reg = float(np.corrcoef(preds_reg, y_test_reg_seq)[0, 1])

In [ ]:
# Plot forward-return regression diagnostics using the configured symbol and
# horizon.
fig, axes = plt.subplots(2, 1, figsize=(9.5, 5.5))
ax = axes[0]
ax.plot(
    dates_test_reg,
    y_test_reg_seq * 100.0,
    label="True",
    color="tab:blue",
    linewidth=0.8,
)
ax.plot(
    dates_test_reg,
    preds_reg * 100.0,
    label="Predicted",
    color="tab:red",
    linewidth=1.0,
    alpha=0.8,
)
ax.set_ylabel(f"{HORIZON_REG}-step return (%)")
ax.set_title(
    f"{SYMBOL_REG} {HORIZON_REG}-step return regression "
    f"(MSE={mse_reg:.5f}, corr={corr_reg:.2f})"
)
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.3)
ax = axes[1]
true_pct = y_test_reg_seq * 100.0
pred_pct = preds_reg * 100.0
ax.scatter(
    true_pct,
    pred_pct,
    color="tab:purple",
    alpha=0.7,
    edgecolor="k",
    linewidth=0.3,
)
combined = np.concatenate([true_pct, pred_pct])
max_abs = float(np.percentile(np.abs(combined), 99.0))
max_abs = max(max_abs, 1.0)
lims = np.array([-max_abs, max_abs])
ax.set_xlim(-max_abs, max_abs)
ax.set_ylim(-max_abs, max_abs)
ax.plot(lims, lims, "k--", linewidth=1.0, alpha=0.7)
ax.set_xlabel(f"True {HORIZON_REG}-step return (%)")
ax.set_ylabel(f"Predicted {HORIZON_REG}-step return (%)")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Compare equity curves implied by the realised and predicted forward returns.
# Use a stride of `HORIZON_REG` to avoid overlapping multi-step returns
# when you change the horizon.
stride_reg = max(int(HORIZON_REG), 1)
dates_eq = dates_test_reg[::stride_reg]
eq_true = np.exp(np.cumsum(y_test_reg_seq[::stride_reg]))
eq_pred = np.exp(np.cumsum(preds_reg[::stride_reg]))
eq_true /= eq_true[0]
eq_pred /= eq_pred[0]
fig_eq, ax_eq = plt.subplots(1, 1, figsize=(9.5, 3.0))
ax_eq.plot(
    dates_eq,
    eq_true,
    label="True equity curve",
    color="tab:blue",
    linewidth=1.0,
)
ax_eq.plot(
    dates_eq,
    eq_pred,
    label="Predicted equity curve",
    color="tab:red",
    linewidth=1.0,
    alpha=0.9,
)
ax_eq.set_ylabel("Equity (normalised to 1)")
ax_eq.set_xlabel("Date")
ax_eq.set_title(f"{SYMBOL_REG} equity curves from true vs predicted returns")
ax_eq.legend(loc="upper left")
ax_eq.grid(True, linestyle="--", alpha=0.3)
fig_eq.tight_layout()
plt.show()

## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
